# Download the non-blind ICLR2026 data from OpenReview

In [1]:
import numpy as np
import pandas as pd
import pylab as plt

import requests
import time

In [2]:
# Load iclr2026v1

iclr_old = pd.read_parquet('../data/iclr26v1.parquet')

iclr_old

,year,id,title,abstract,authors,author_ids,decision,scores,keywords,labels
0,2017,B1-Hhnslg,Prototypical Networks for Few-shot Learning,A recent approach to few-shot classification c...,"Jake Snell, Kevin Swersky, Richard Zemel",,Reject,"[6, 4, 5]","[deep learning, transfer learning]",transfer learning
1,2017,B1-q5Pqxl,Machine Comprehension Using Match-LSTM and Ans...,Machine comprehension of text is an important ...,"Shuohang Wang, Jing Jiang",,Accept (Poster),"[6, 6, 7]","[natural language processing, deep learning]",language models
2,2017,B16Jem9xe,Learning in Implicit Generative Models,Generative adversarial networks (GANs) provide...,"Shakir Mohamed, Balaji Lakshminarayanan",,Invite to Workshop Track,"[8, 7, 6]",[unsupervised learning],unlabeled
3,2017,B16dGcqlx,Third Person Imitation Learning,Reinforcement learning (RL) makes it possible ...,"Bradly C Stadie, Pieter Abbeel, Ilya Sutskever",,Accept (Poster),"[6, 5, 6]",[],unlabeled
4,2017,B184E5qee,Improving Neural Language Models with a Contin...,We propose an extension to neural network lang...,"Edouard Grave, Armand Joulin, Nicolas Usunier",,Accept (Poster),"[7, 9, 5]",[natural language processing],language models
...,...,...,...,...,...,...,...,...,...,...
55901,2026,zz3El6hqbs,Learning activation functions with PCA on a se...,This work explores a novel approach to learnin...,,,,[],"[deep neural networks, activation function lea...",unlabeled
55902,2026,zzJTo7ujql,Phased DMD: Few-step Distribution Matching Dis...,Distribution Matching Distillation (DMD) disti...,,,,[],"[diffusion models, distribution matching, dist...",diffusion models
55903,2026,zzTDulLys0,vAttention: Verified Sparse Attention via Samp...,State-of-the-art sparse attention methods for ...,,,,[],[sparse attention],unlabeled
55904,2026,zzTQISAGUp,Polychromic Objectives for Reinforcement Learning,Reinforcement learning fine-tuning (RLFT) is a...,,,,[],"[reinforcement learning, exploration]",RL


In [3]:
%%time

# Download titles/abstracts/authors of all papers
# We are doing it here to disambiguate the authors by author IDs

titles = []
abstracts = []
years = []
forum_ids = []
decisions = []
authors = []
author_ids = []
keywords = []

for year in [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]:
    print(year, end=': ')
    for query in ['submission', 'Submission', 'Blind_Submission', 
                  'Withdrawn_Submission', 'Rejected_Submission', 
                  'Desk_Rejected_Submission', '']:
        if year <= 2017:
            if query == '':
                continue
            url = f'https://api.openreview.net/notes?invitation=ICLR.cc%2F{year}%2Fconference%2F-%2F{query}'
        elif year <= 2023:
            if query == '':
                continue
            url = f'https://api.openreview.net/notes?invitation=ICLR.cc%2F{year}%2FConference%2F-%2F{query}'
        else:
            if query != '':
                query = '/' + query
            url = f'https://api2.openreview.net/notes?content.venueid=ICLR.cc/{year}/Conference{query}'        
            
        for offset in range(0, 20_000, 1000):
            json = requests.get(url + f'&offset={offset}').json()
            
            if 'name' in json and json['name'] == 'RateLimitError':
                time.sleep(30)
                json = requests.get(url + f'&offset={offset}').json()
                
            df = pd.DataFrame(json['notes'])
            if len(df) > 0:
                print(len(df), end=' ')
                if year < 2024:
                    titles    += [d['title'].strip() for d in df['content'].values]
                    abstracts += [d['abstract'].strip() for d in df['content'].values]
                    keywords  += [d['keywords'] for d in df['content'].values]
                    if year == 2017:
                        authors   += [', '.join(d['authors']) if isinstance(d['authors'], list)
                                      else d['authors'] for d in df['content']]
                        author_ids = [', '.join(d['authorids']) if 'authorids' in d 
                                      else d['author_emails'] for d in df['content']]
                    else:
                        authors   += [', '.join(d['authors']) for d in df['content']]
                        author_ids  += [', '.join(d['authorids']) for d in df['content']]
                else:
                    titles    += [d['title']['value'].strip() for d in df['content'].values]
                    abstracts += [d['abstract']['value'].strip() for d in df['content'].values]
                    keywords  += [d['keywords']['value'] for d in df['content'].values]
                    if 'authors' in df['content'].values[0]:
                        authors   += [', '.join(d['authors']['value']) for d in df['content'].values]
                        author_ids   += [', '.join(d['authorids']['value']) for d in df['content'].values]
                    else:
                        authors += [''] * len(df)
                        author_ids += [''] * len(df)
                years     += [year] * len(df)
                forum_ids += list(df.forum)
                                                
                if 'Withdrawn_Submission' in query:
                    decisions += ['Withdrawn'] * len(df)
                elif 'Desk_Rejected_Submission' in query:
                    decisions += ['Desk rejected'] * len(df)
                elif 'Rejected_Submission' in query:
                    decisions += ['Reject'] * len(df)    
                else:
                    decisions += [''] * len(df)
            else:
                break
    print('')
print('')

print(f'Found {len(titles)} papers\n')

2017: 490 
2018: 935 83 
2019: 1000 419 160 
2020: 1000 1000 213 369 12 
2021: 1000 1000 594 403 17 
2022: 1000 1000 617 779 26 
2023: 1000 1000 1000 792 1000 145 18 
2024: 1000 659 1000 1000 1000 432 53 1000 1000 260 
2025: 1000 1000 987 1000 1000 1000 1000 912 70 1000 1000 1000 703 
2026: 1000 1000 1000 1000 1000 122 1000 1000 1000 1000 1000 1000 1000 1000 468 1000 1000 1000 1000 1000 359 

Found 55097 papers

CPU times: user 4.74 s, sys: 280 ms, total: 5.02 s
Wall time: 2min 51s


In [4]:
# Split keyword strings with semicolons instead of commas
keywords = [
    [kk.strip() for kk in k[0].split(";") if kk.strip() != ''] 
    if len(k) > 0 and ";" in k[0] else k
    for k in keywords
]

# Remove trailing periods from keywords
keywords = [
    k[:-1] + [k[-1].strip()[:-1]]
    if len(k) > 0 and "." in k[-1].strip()[-1] else k
    for k in keywords
]

# Make sure all is lower-case
keywords = [[kk.lower() for kk in k] for k in keywords]

iclr = pd.DataFrame.from_dict({
    'year': years,
    'id': forum_ids, 
    'title': titles,
    'abstract': abstracts,
    'authors': authors,
    'author_ids': author_ids,
    'decision': decisions,
    'scores': [[]] * len(forum_ids),
    'keywords': keywords,
    'labels': [''] * len(forum_ids)
})

# Removing author IDs for papers <= 2020 because emails were used as IDs
# and some authors had missing emails

iclr.loc[iclr.year <= 2020, 'author_ids'] = ''

# Kicking out nonsense abstracts

n_submissions = [np.sum(iclr.year == y) for y in np.arange(2017, 2027)]
print('Submissions per year:', n_submissions, '\n')

mask = np.array([len(a) >= 100 for a in iclr.abstract])

print(f'Removing {np.sum(~mask)} submissions with abstract length below 100 characters\n')

iclr = iclr[mask].reset_index(drop=True)

n_submissions = [np.sum(iclr.year == y) for y in np.arange(2017, 2027)]
print('Submissions per year:', n_submissions, '\n')

print('Dataset size:', len(iclr))

# Sort by year and id

iclr = iclr.sort_values(by=['year', 'id']).reset_index(drop=True)

iclr

Submissions per year: [490, 1018, 1579, 2594, 3014, 3422, 4955, 7404, 11672, 18949] 

Removing 50 submissions with abstract length below 100 characters

Submissions per year: [489, 1012, 1569, 2593, 3009, 3422, 4955, 7401, 11663, 18934] 

Dataset size: 55047


,year,id,title,abstract,authors,author_ids,decision,scores,keywords,labels
0,2017,B1-Hhnslg,Prototypical Networks for Few-shot Learning,A recent approach to few-shot classification c...,"Jake Snell, Kevin Swersky, Richard Zemel",,,[],"[deep learning, transfer learning]",
1,2017,B1-q5Pqxl,Machine Comprehension Using Match-LSTM and Ans...,Machine comprehension of text is an important ...,"Shuohang Wang, Jing Jiang",,,[],"[natural language processing, deep learning]",
2,2017,B16Jem9xe,Learning in Implicit Generative Models,Generative adversarial networks (GANs) provide...,"Shakir Mohamed, Balaji Lakshminarayanan",,,[],[unsupervised learning],
3,2017,B16dGcqlx,Third Person Imitation Learning,Reinforcement learning (RL) makes it possible ...,"Bradly C Stadie, Pieter Abbeel, Ilya Sutskever",,,[],[],
4,2017,B184E5qee,Improving Neural Language Models with a Contin...,We propose an extension to neural network lang...,"Edouard Grave, Armand Joulin, Nicolas Usunier",,,[],[natural language processing],
...,...,...,...,...,...,...,...,...,...,...
55042,2026,zz3El6hqbs,Learning activation functions with PCA on a se...,This work explores a novel approach to learnin...,,,Reject,[],"[deep neural networks, activation function lea...",
55043,2026,zzJTo7ujql,Phased DMD: Few-step Distribution Matching Dis...,Distribution Matching Distillation (DMD) disti...,"Xiangyu Fan, Zesong Qiu, Zhuguanyu Wu, Fanzhou...","~Xiangyu_Fan3, ~Zesong_Qiu1, ~Zhuguanyu_Wu1, ~...",Withdrawn,[],"[diffusion models, distribution matching, dist...",
55044,2026,zzTDulLys0,vAttention: Verified Sparse Attention via Samp...,State-of-the-art sparse attention methods for ...,,,,[],[sparse attention],
55045,2026,zzTQISAGUp,Polychromic Objectives for Reinforcement Learning,Reinforcement learning fine-tuning (RLFT) is a...,,,,[],"[reinforcement learning, exploration]",


In [5]:
# Copy decisions/scores/labels from iclr26v1

for i, idd in enumerate(iclr.id):
    if idd in iclr_old.id.values:
        iclr_old_pos = np.where(iclr_old.id.values == idd)[0][0]
        iclr.at[i, 'decision'] = iclr_old.at[iclr_old_pos, 'decision']
        iclr.at[i, 'scores'] = iclr_old.at[iclr_old_pos, 'scores']
        iclr.at[i, 'labels'] = iclr_old.at[iclr_old_pos, 'labels']

# Mark papers found in pre-2026 years but missing from prior version as Withdrawn
missing_info = set(iclr[iclr.year < 2026].id) - set(iclr_old[iclr_old.year < 2026].id)
iclr.loc[iclr.id.isin(missing_info), 'decision'] = 'Withdrawn'

iclr

,year,id,title,abstract,authors,author_ids,decision,scores,keywords,labels
0,2017,B1-Hhnslg,Prototypical Networks for Few-shot Learning,A recent approach to few-shot classification c...,"Jake Snell, Kevin Swersky, Richard Zemel",,Reject,"[6, 4, 5]","[deep learning, transfer learning]",transfer learning
1,2017,B1-q5Pqxl,Machine Comprehension Using Match-LSTM and Ans...,Machine comprehension of text is an important ...,"Shuohang Wang, Jing Jiang",,Accept (Poster),"[6, 6, 7]","[natural language processing, deep learning]",language models
2,2017,B16Jem9xe,Learning in Implicit Generative Models,Generative adversarial networks (GANs) provide...,"Shakir Mohamed, Balaji Lakshminarayanan",,Invite to Workshop Track,"[8, 7, 6]",[unsupervised learning],unlabeled
3,2017,B16dGcqlx,Third Person Imitation Learning,Reinforcement learning (RL) makes it possible ...,"Bradly C Stadie, Pieter Abbeel, Ilya Sutskever",,Accept (Poster),"[6, 5, 6]",[],unlabeled
4,2017,B184E5qee,Improving Neural Language Models with a Contin...,We propose an extension to neural network lang...,"Edouard Grave, Armand Joulin, Nicolas Usunier",,Accept (Poster),"[7, 9, 5]",[natural language processing],language models
...,...,...,...,...,...,...,...,...,...,...
55042,2026,zz3El6hqbs,Learning activation functions with PCA on a se...,This work explores a novel approach to learnin...,,,,[],"[deep neural networks, activation function lea...",unlabeled
55043,2026,zzJTo7ujql,Phased DMD: Few-step Distribution Matching Dis...,Distribution Matching Distillation (DMD) disti...,"Xiangyu Fan, Zesong Qiu, Zhuguanyu Wu, Fanzhou...","~Xiangyu_Fan3, ~Zesong_Qiu1, ~Zhuguanyu_Wu1, ~...",,[],"[diffusion models, distribution matching, dist...",diffusion models
55044,2026,zzTDulLys0,vAttention: Verified Sparse Attention via Samp...,State-of-the-art sparse attention methods for ...,,,,[],[sparse attention],unlabeled
55045,2026,zzTQISAGUp,Polychromic Objectives for Reinforcement Learning,Reinforcement learning fine-tuning (RLFT) is a...,,,,[],"[reinforcement learning, exploration]",RL


In [11]:
%%time

# Query the accept/reject decisions and scores for 2026 papers
# API cuts you off every 60 queries, then the code sleeps for 30 seconds
def request_with_retry(forum_id):
    url = f'https://api2.openreview.net/notes?forum={forum_id}'
    json = requests.get(url).json()

    def is_rate_limit_error(json):
        return 'name' in json and json['name'] == 'RateLimitError'

    if is_rate_limit_error(json):
        time.sleep(10)
        # recursively retry until we get a non-rate-limit-error response
        # previous code had 30sec wait here and then retry only once, but that would sometimes lead to a rate limit error again
        print(f'Rate limit error for forum_id {forum_id}, retrying in 10sec...')
        return request_with_retry(forum_id)

    return json


for num, forum_id in enumerate(iclr.id):
    if iclr.year[num] < 2026:
        continue
    
    if (num + 1) % 1000 == 0:
        print('*', end='')
    elif (num + 1) % 100 == 0:
        print('.', end='')

    year = iclr.year[num]
    
    json = request_with_retry(forum_id)

    found_decision = False
    for i in range(len(json['notes'])):
        if 'decision' in json['notes'][i]['content']:
            decision = json['notes'][i]['content']['decision']['value']
            found_decision = True
            break
        if 'withdrawal_confirmation' in json['notes'][i]['content']:
            decision = 'Withdrawn'
            found_decision = True
            break
        if 'desk_reject_comments' in json['notes'][i]['content']:
            decision = 'Desk rejected'
            found_decision = True
            break
    if found_decision:
        iclr.at[num, 'decision'] = decision
    else:
        print(f'No decision found: {num}, {forum_id}')
        
    scores = []
    for i in range(len(json['notes'])):
        if 'rating' in json['notes'][i]['content']:
            score = int(json['notes'][i]['content']['rating']['value'])
            scores.append(score)
    iclr.at[num, 'scores'] = scores

print('')

iclr.to_parquet('../data/iclr26v2.parquet')

Rate limit error for forum_id 0BkvUY61MX, retrying in 10sec...
Rate limit error for forum_id 0BkvUY61MX, retrying in 10sec...
Rate limit error for forum_id 0BkvUY61MX, retrying in 10sec...
.Rate limit error for forum_id 0OUkySEAf0, retrying in 10sec...
Rate limit error for forum_id 0OUkySEAf0, retrying in 10sec...
Rate limit error for forum_id 0OUkySEAf0, retrying in 10sec...
Rate limit error for forum_id 0Z9DZtIZI7, retrying in 10sec...
Rate limit error for forum_id 0Z9DZtIZI7, retrying in 10sec...
Rate limit error for forum_id 0Z9DZtIZI7, retrying in 10sec...
.No decision found: 36305, 0aTSKdkC2s
Rate limit error for forum_id 0ifNNBqBlQ, retrying in 10sec...
Rate limit error for forum_id 0ifNNBqBlQ, retrying in 10sec...
Rate limit error for forum_id 0ifNNBqBlQ, retrying in 10sec...
No decision found: 36396, 0qVrS2XdOW
.Rate limit error for forum_id 0unRcs07Bb, retrying in 10sec...
Rate limit error for forum_id 0unRcs07Bb, retrying in 10sec...
Rate limit error for forum_id 0unRcs07Bb,

In [13]:
iclr[iclr.year == 2026]

,year,id,title,abstract,authors,author_ids,decision,scores,keywords,labels
36113,2026,00F7BfXLYJ,CyberV: A Cybernetic Framework for Enhancing L...,Current Multimodal Large Language Models (MLLM...,"Jiahao Meng, Shuyang Sun, Tan Yue, Lu Qi, Yunh...","~Jiahao_Meng1, ~Shuyang_Sun1, ~Tan_Yue2, ~Lu_Q...",Withdrawn,"[4, 4, 4, 4]","[video understanding, multimodal large languag...",unlabeled
36114,2026,00HNN8O7Ni,Learning Reactive Synthesis from Model Checkin...,Deep learning applications to formal verificat...,,,Reject,"[4, 2, 2, 4]","[temporal logic, reactive synthesis, expert it...",unlabeled
36115,2026,00UQtHqB2k,Toward Unifying Group Fairness Evaluation from...,Ensuring algorithmic fairness remains a signif...,,,Reject,"[4, 2, 6, 2]","[fairness, sparsity, unified framework]",fairness
36116,2026,017F77AYeQ,SMART-3D: Scaling Masked AutoRegressive Transf...,Autoregressive models have shown promise in 3D...,"Shentong Mo, Yufei Guo","~Shentong_Mo1, ~Yufei_Guo1",Withdrawn,"[0, 4, 2, 2]","[autoregressive models, 3d shape generation]",unlabeled
36117,2026,023yMrtHQP,Expectation–Evidence Prompting: Structuring Ve...,Large language models (LLMs) often fail in fac...,"Chang Wang, Longwei Wang, KC Santosh, Chaowei ...","~Chang_Wang13, ~Longwei_Wang1, ~KC_Santosh1, ~...",Withdrawn,"[4, 4, 4]","[large language models (llms), factual verific...",unlabeled
...,...,...,...,...,...,...,...,...,...,...
55042,2026,zz3El6hqbs,Learning activation functions with PCA on a se...,This work explores a novel approach to learnin...,,,Reject,"[2, 2, 8, 2]","[deep neural networks, activation function lea...",unlabeled
55043,2026,zzJTo7ujql,Phased DMD: Few-step Distribution Matching Dis...,Distribution Matching Distillation (DMD) disti...,"Xiangyu Fan, Zesong Qiu, Zhuguanyu Wu, Fanzhou...","~Xiangyu_Fan3, ~Zesong_Qiu1, ~Zhuguanyu_Wu1, ~...",Withdrawn,"[4, 2, 4]","[diffusion models, distribution matching, dist...",diffusion models
55044,2026,zzTDulLys0,vAttention: Verified Sparse Attention via Samp...,State-of-the-art sparse attention methods for ...,,,Accept (Poster),"[6, 6, 6, 2]",[sparse attention],unlabeled
55045,2026,zzTQISAGUp,Polychromic Objectives for Reinforcement Learning,Reinforcement learning fine-tuning (RLFT) is a...,,,Accept (Poster),"[6, 8, 6, 2]","[reinforcement learning, exploration]",RL


In [8]:
[np.sum(iclr.year == y) for y in np.arange(2017, 2027)]

[489, 1012, 1569, 2593, 3009, 3422, 4955, 7401, 11663, 18934]